# Facies Classification from Well Log Data
## Machine learning for energy applications — Module 1, Exercise 4

In this exercise we build a supervised classification model to predict geological **facies** from well log measurements. Facies classification is a core task in subsurface characterization: petrophysicists and geologists use wireline log responses (gamma ray, resistivity, porosity, photoelectric effect, etc.) to identify rock types encountered during drilling.

The dataset originates from a carbonate reservoir study in the Hugoton gas field, Kansas. Nine distinct facies have been labelled by an expert interpreter. Our goal is to train a Random Forest classifier that can reproduce those labels given the log curves alone, **in a well it has never seen**.

### Learning objectives
1. Load, merge and explore a real well-log dataset, and audit its data quality well by well.
2. Visualize feature distributions and class balance.
3. Prepare data — apply fixed cleaning rules and engineer features before the split; keep fitted steps (imputation, encoding) inside a pipeline.
4. Hold out one whole well as the test set and compare a Random Forest with a majority-class baseline on cross-validation folds made of whole wells.
5. Tune hyperparameters with GridSearchCV on the same well-grouped folds.
6. Open the test well once, for the locked model, and interpret the model through feature importances and a confusion matrix.

---
## 1. Import Libraries

Import the following libraries: `numpy`, `pandas`, `matplotlib.pyplot`, `seaborn`, and from scikit-learn: `clone`, `ColumnTransformer`, `DummyClassifier`, `RandomForestClassifier`, `SimpleImputer`, `GridSearchCV`, `GroupKFold`, `KFold`, `cross_validate`, `Pipeline`, `OrdinalEncoder`, `accuracy_score`, `classification_report`, `confusion_matrix`, and `f1_score`.

In [ ]:
# TODO: Import all required libraries

---
## 2. Load and Explore the Data

We have two CSV files:
- **facies_dataset.csv** — 4,149 depth samples from 10 wells, each with wireline log measurements and an expert-assigned facies label. Depths are in metres (as the `Description` column states), with samples nominally 0.5 m apart within a well; confirm the units with the data owner before building depth-derived features.
- **facies_labels.csv** — a lookup table mapping the integer facies code (1–9) to a descriptive rock-type name.

After loading we merge them so that every sample carries both the numeric code and the human-readable facies name.

**Tasks:**
1. Read both CSV files with `pd.read_csv()`.
2. Print the shape and column names of the main dataset.
3. Display the first few rows with `.head()`.

In [ ]:
# TODO: Load facies_dataset.csv and facies_labels.csv
# Print shape, columns, and show head()


In [ ]:
# TODO: Merge the labels into the main dataframe on the 'Facies' column
# Display the merged result

In [ ]:
# TODO: Show descriptive statistics with .describe()
# Check for missing values with .isnull().sum()
# Print the unique well names


### 2.1 Audit by Well

Missing values and implausible readings are rarely spread evenly: they come with particular wells, tools or logging runs. Before any modelling, count them **per well**, and compare each curve's maximum with its median. PE (photoelectric factor) above 20 and GR above 200 API deserve a question to the data owner.

**Task:** Build an audit table with one row per well: number of rows, % missing GR, % missing PE, count of GR > 200 API, count of PE > 20, whether the depths are sorted, and the number of repeated depths. Print the median and maximum of GR and PE.

In [ ]:
# TODO: Build the audit table, one row per well
# Hint: by_well = df.groupby("Well Name", sort=False), then e.g.
#       by_well["PE"].apply(lambda s: 100 * s.isna().mean()) for % missing PE
# Print the median and maximum of GR and PE

---
## 3. Visualize the Data

Before modelling it is important to understand class balance and inter-feature relationships. In well-log interpretation, certain facies are inherently rarer (e.g., phylloid-algal bafflestone), which affects classifier performance.

These plots describe all ten wells, including the one held out as the test well in section 6. That is acceptable for describing the data, but no modelling choice below is taken from them; in a project, set the test well aside before looking at its labels.

**Tasks:**
1. Create a bar chart showing the distribution (count) of each facies type.
2. Create a pair plot of the key wireline features (`GR`, `ILD_log10`, `DeltaPHI`, `PHIND`) coloured by facies type.

In [ ]:
# TODO: Create a bar chart of facies type counts
# Hint: Use df['Facies_type'].value_counts() and .plot(kind='bar')


In [ ]:
# TODO: Create a pair plot of GR, ILD_log10, DeltaPHI, PHIND coloured by Facies_type
# Hint: Use sns.pairplot() with hue='Facies_type'


---
## 4. Data Preparation

We need to:
1. **Drop non-predictive columns** — `Well Name` is an identifier, not a feature (we keep a copy as `Well` for the train/test split and the grouped folds). The `Description` column is auto-generated text that repeats the well name, depth, formation and GR: it carries no additional signal and would expose the well identity. `Facies_type` is the label in words.
2. **Apply fixed cleaning rules** — PE values above 20 are recording errors in this file (every other PE value is below 8.1), so we set them to missing. A fixed rule learns nothing from the data, so it may run before the split. GR above 200 API can be real in radioactive shales; we keep those values and flag them for the data owner.
3. **Impute missing values inside the pipeline** — Median imputation learns medians, so it is fitted later, inside the model pipeline, on training wells only. PE is missing in two whole wells, so the imputer also adds missing-value indicator columns: the model can tell an imputed PE from a measured one.
4. **Encode categorical features inside the pipeline** — `Formation` is a geological zone name. `OrdinalEncoder` gives each zone an integer code; it learns the list of zones, so it is fitted inside the pipeline too. Integer codes are acceptable for a tree-based model like the Random Forest, which can isolate any code with a few splits, but not for distance-based models such as KNN, which would read the arbitrary codes as distances; those need one-hot encoding.

**Tasks:**
1. Copy `Well Name` to a new column `Well` (we need it later for the split and the grouped folds), then drop `Well Name`, `Description`, and `Facies_type`.
2. Set `PE` values above 20 to missing.
3. Do **not** impute or encode yet: print the missing-value counts that the pipeline will handle.

In [ ]:
# TODO: Prepare the data
# 1. Copy Well Name to a 'Well' column, then drop Well Name, Description and Facies_type
# 2. Set PE values above 20 to missing (np.nan)
# 3. Print the missing-value counts left for the pipeline (do not impute here)

---
## 5. Feature Engineering

Domain knowledge can improve model performance:
- **GR bins** — Gamma ray is often interpreted in broad ranges (low = clean sand, high = shale). We discretize it into bins with fixed edges at 45, 75 and 150 API.
- **Depth-normalized features** — Normalizing GR and PHIND by depth can capture trends related to burial compaction. A ratio to depth changes with the depth unit (metres here), which is one more reason to confirm units.

Both are row-by-row formulas with fixed constants, so computing them before the split does not leak information between wells. Missing GR stays missing in the new columns and is imputed inside the pipeline.

**Tasks:**
1. Create a `GR_bin` column using `pd.cut()` with the fixed edges `[-np.inf, 45, 75, 150, np.inf]` and labels `[0, 1, 2, 3]`.
2. Create `GR_depth_norm` (= GR / Depth) and `PHIND_depth_norm` (= PHIND / Depth).

In [ ]:
# TODO: Engineer new features
# 1. GR_bin using pd.cut with bins [-np.inf, 45, 75, 150, np.inf] and labels [0,1,2,3]
# 2. GR_depth_norm = GR / Depth
# 3. PHIND_depth_norm = PHIND / Depth

---
## 6. Train / Test Split — Leave-One-Well-Out

In subsurface ML a random train/test split would leak spatial information because adjacent depth samples in the same well are highly correlated. Instead we hold out an entire well for testing. This is the **leave-one-well-out** strategy and gives a much more realistic estimate of how the model will perform on a new, unseen well.

We use the **SHANKLE** well as our test set. It stays closed until section 10: every choice before that (the baseline comparison and the tuning) uses the nine training wells only, with cross-validation folds made of whole wells.

**Tasks:**
1. Define feature columns (everything except `Facies` and `Well`); `Formation` is the only categorical column.
2. Create boolean masks for train (Well != 'SHANKLE') and test (Well == 'SHANKLE').
3. Build `X_train`, `y_train`, `groups_train` (the well of each training row), `X_test` and `y_test`, and print their sizes. Keep `X_train` and `X_test` as DataFrames so the pipeline can select columns by name.

In [ ]:
# TODO: Split data using leave-one-well-out with SHANKLE as test well
# Keep groups_train (the Well of each training row) for the grouped folds
# Print training/test sample counts and feature list

---
## 7. Default Random Forest Against a Majority-Class Baseline

Imputation and encoding go into one pipeline with the model, so every fit — in each cross-validation fold and in the final refit — learns its medians and formation codes from its own training wells only.

We compare a default-parameter Random Forest with a **majority-class baseline** (always predict the most common facies) on the same folds. The folds are made of whole training wells (`GroupKFold`, three wells per validation fold), so each fold asks the deployment question: how well does the model do on wells it has not seen? For contrast only, we also score the forest with shuffled row folds, which put neighbouring samples of the same well on both sides of every split. We report macro F1, which weights every facies equally, and accuracy. The test well is not used here.

**Tasks:**
1. Build a `ColumnTransformer` with `SimpleImputer(strategy="median", add_indicator=True)` for the numeric columns and `OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)` for `Formation`, and put it in a `Pipeline` with each model.
2. Score a majority-class baseline (`DummyClassifier(strategy="most_frequent")`) and a default `RandomForestClassifier(n_estimators=100, random_state=42)` with `cross_validate` on `GroupKFold(n_splits=3)`, passing `groups=groups_train`; use `scoring=["f1_macro", "accuracy"]`. Print the validation wells of each fold.
3. For contrast, score the forest with `KFold(n_splits=5, shuffle=True, random_state=42)` row folds. Which number answers the new-well question?
4. Fit the default forest pipeline on all nine training wells (for section 8) and print the Formation codes its encoder learned. Do not score it on the test well.

In [ ]:
# TODO: Build the preprocessing ColumnTransformer and a Pipeline for each model
# TODO: Compare a majority-class baseline and a default Random Forest with cross_validate
#       on GroupKFold(n_splits=3) folds (groups=groups_train); print the validation wells per fold
# TODO: For contrast, score the forest with shuffled KFold(n_splits=5) row folds
# TODO: Fit the default forest pipeline on all training wells (not on the test well)
#       and print the Formation codes its encoder learned (categories_)

---
## 8. Feature Importance

Random Forests provide a built-in measure of feature importance (mean decrease in impurity). This helps petrophysicists understand which log curves the model relies on most — and whether that aligns with domain expectations. These importances describe how the forest splits the training wells; they are not causal effects, and they tend to favour features with many distinct values. Module 2 compares them with permutation importance.

**Task:** Create a horizontal bar chart of feature importances from the default forest pipeline, sorted from lowest to highest. Use `pipeline.named_steps["prep"].get_feature_names_out()` for the names, including the missing-value indicators.

In [ ]:
# TODO: Plot feature importances as a horizontal bar chart
# Hint: use pipeline.named_steps["model"].feature_importances_ with
#       pipeline.named_steps["prep"].get_feature_names_out(), and pd.Series(...).sort_values()

---
## 9. Hyperparameter Tuning with GridSearchCV

We search over a grid of key Random Forest parameters:
- **n_estimators** — number of trees (more trees = smoother predictions but longer training).
- **max_depth** — controls how deep each tree can grow (limits overfitting).
- **min_samples_split** — minimum samples required to split a node.

The folds follow wells, as in section 7: `GroupKFold(n_splits=3)` on the nine training wells, with `groups=groups_train` passed to `fit`. A row-level `cv=5` would mix samples of the same well across training and validation folds, so it would not measure performance on unseen wells. We select on macro F1 so that rare facies count. The best cross-validated score is the maximum of 18 noisy numbers, so it is optimistic; compare the differences between settings with the fold-to-fold spread. `GridSearchCV` then refits the chosen settings on all nine training wells. The test well is still closed.

**Tasks:**
1. Define a `param_grid` with `model__n_estimators=[50,100,200]`, `model__max_depth=[None,10,20]`, `model__min_samples_split=[2,5]` (the `model__` prefix addresses the forest inside the pipeline).
2. Run `GridSearchCV` on the forest pipeline with `cv=GroupKFold(n_splits=3)` and `scoring='f1_macro'`, and pass `groups=groups_train` to `fit`.
3. Print the best grouped-CV score, its three fold scores, and the best parameters.

In [ ]:
# TODO: Perform GridSearchCV with the specified parameter grid on GroupKFold(n_splits=3) folds
# Remember to pass groups=groups_train to fit
# Print the best grouped-CV macro F1, its fold scores and the best parameters

---
## 10. Confusion Matrix

Now, and only now, we open the test well: the tuned pipeline is locked, and we score it on SHANKLE once. The confusion matrix reveals which facies pairs the model confuses most. Neighbouring facies with overlapping log responses are the usual confusions in the Hugoton data, for example the nonmarine sandstone and the two nonmarine siltstones, or wackestone and packstone-grainstone. Look for the largest off-diagonal cells in this well.

**Tasks:**
1. Use the best estimator from GridSearchCV (the locked pipeline) to predict the test well — the first and only time the test well is used.
2. Compute the confusion matrix.
3. Plot it as a heatmap using `sns.heatmap()` with facies names as tick labels.

In [ ]:
# TODO: Open the test well once: predict it with the locked (tuned) pipeline
# Plot the confusion matrix heatmap, using facies names as axis labels

---
## 11. Final Evaluation

Summarize the results: print the best hyperparameters, the best grouped cross-validation score (a validation estimate, optimistic because it is the best of 18), and the test-well accuracy and macro F1 of the locked pipeline next to the majority-class baseline on the same test rows. Also print the full classification report for the locked pipeline. One well is one noisy sample of the new-well question: report the score as such, and do not go back and re-tune on it.

In [ ]:
# TODO: Print final evaluation summary
# Include: best parameters, best grouped-CV score, and the test-well accuracy and macro F1
#          of the locked pipeline and of the majority-class baseline
# Print classification_report for the locked pipeline

---
## Summary and Next Steps

In this exercise we:
- Loaded and merged well-log data with facies labels from the Hugoton gas field, and audited missing and implausible values well by well.
- Explored class balance and feature relationships through visualizations.
- Applied a fixed cleaning rule and row-wise engineered features before the split, and kept every fitted step (median imputation with missing-value indicators, formation encoding) inside a pipeline.
- Held out one whole well (**leave-one-well-out**) as the test set and compared a default Random Forest with a majority-class baseline on folds made of whole training wells.
- Tuned the forest with GridSearchCV on the same well-grouped folds, then opened the test well once for the locked pipeline.
- Evaluated with accuracy, macro F1, classification reports, and a confusion matrix.

**Ideas for further exploration:**
- Try gradient-boosted trees (XGBoost, LightGBM) — they often outperform Random Forests on tabular data; compare them on the same well-grouped folds.
- Experiment with additional engineered features (e.g., rolling averages of GR over a depth window to capture stratigraphic trends), computed within each well.
- Repeat the whole procedure with each well as the test well in turn (nested cross-validation over wells) to see how much the score varies between wells.
- Address class imbalance with class-weight adjustments, or with resampling such as SMOTE applied inside the training folds only.